# Reproducibility smoke test — 32-subject sample

This notebook runs `../validation/benchmark_script.py` — **unmodified, the exact
script used for the published benchmark** — against the 32-subject sample shipped
in this repository (`../sample/`).

**Purpose:** prove the code runs end-to-end against the real delivered schema.

**Not the purpose:** reproduce the published metric. With `n=32` and 5-fold
cross-validation, each test fold has ~6 subjects. AUC/accuracy computed on 6
points is dominated by noise, not signal — the numbers below will swing wildly
on a different seed or a different 6 subjects, and **must not be quoted as the
dataset's benchmark**. The real, statistically meaningful number is reported on
the full 10,000-subject dataset (see repo root `README.md` — Task B, AUC = 0.779,
5-fold CV, logistic regression, n=10,000).

In [1]:
import sys, json
sys.path.insert(0, "../validation")
import importlib.util
spec = importlib.util.spec_from_file_location("benchmark_script", "../validation/benchmark_script.py")
benchmark_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark_script)

benchmark_script.DATASET = "../sample"
benchmark_script.OUT = "./smoke_test_results.parquet"

summary = benchmark_script.main()


{
 "task_A_detection": {
  "description": "Predict presyncope from DURING-THE-TEST observations (BP/HR readings + crew covariates) -- NOT from truth. Near-perfect by construction: presyncope is clinically DEFINED as MAP crossing a threshold, and beat_to_beat_bp is that same signal plus small observation noise, so this mainly confirms the noise model preserves the diagnostic signal, not that a hard prediction problem was solved. Reported honestly rather than hidden (Doctrine Art 9.7/9.8): a 100% number here is expected, not impressive.",
  "n_subjects": 32,
  "n_features": 17,
  "feature_names": [
   "hr_ecg_max",
   "hr_ecg_mean",
   "bp_min",
   "bp_mean",
   "bp_missing_frac",
   "finger_cuff_dropout_frac",
   "symptoms_reported_any",
   "age",
   "fitness",
   "mission_days",
   "prior_flights",
   "cm_compression_garment",
   "cm_fluid_loading",
   "cm_lbnp",
   "cm_none",
   "sex_F",
   "sex_M"
  ],
  "seed": 20261027,
  "mean_auc": 1.0,
  "std_auc": 0.0,
  "mean_accuracy": 1.0,
 

## Reading the output above

- `task_A_detection` (during-test detection) comes out near-perfect, same as on
  the full dataset — expected by construction (see repo README), not a sign of
  anything being solved.
- `task_B_screening` (pre-test crew-covariate screening) is the number that
  matters, and on n=32 it is noisy: `mean_auc` here will typically land
  somewhere in the 0.5–0.9 range purely from which 6 subjects happened to land
  in each test fold, `std_auc` across folds is large relative to the mean, and a
  different `random_state` would move it substantially. None of that is a
  defect in the dataset or the script — it is what 5-fold CV on 32 subjects
  always looks like. It confirms the pipeline runs correctly against this
  exact file layout; it says nothing about the dataset's real predictive power.
- For the real benchmark, see the full dataset (n=10,000): **Task B AUC = 0.779**,
  accuracy = 80.9%, reported in the repo root `README.md` and in the full
  dataset's own `validation/` package.

In [2]:
import pandas as pd
results = pd.read_parquet("./smoke_test_results.parquet")
results


,task,fold,n_test,auc,accuracy,positive_rate_test
0,A_detection_during_test,0,7,1.000000,1.000000,0.571429
1,A_detection_during_test,1,7,1.000000,1.000000,0.428571
2,A_detection_during_test,2,6,1.000000,1.000000,0.500000
3,A_detection_during_test,3,6,1.000000,1.000000,0.500000
4,A_detection_during_test,4,6,1.000000,1.000000,0.500000
5,B_screening_pretest_only,0,7,0.500000,0.714286,0.571429
6,B_screening_pretest_only,1,7,0.500000,0.571429,0.428571
7,B_screening_pretest_only,2,6,0.777778,0.666667,0.500000
8,B_screening_pretest_only,3,6,1.000000,0.666667,0.500000
9,B_screening_pretest_only,4,6,0.666667,0.333333,0.500000
